In [ ]:
import sys, os

import pandas as pd
import numpy as np
from scipy import sparse
from scipy.spatial.distance import pdist
from itertools import combinations

import scanpy as sc
import anndata
from sklearn.linear_model import LinearRegression
from scipy.stats import ks_2samp, ranksums, ttest_ind
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests

import networkx as nx
from umap import UMAP

import matplotlib as mpl
mpl.rc('pdf',fonttype=42)
mpl.rcParams['pdf.use14corefonts'] = True
mpl.rcParams['axes.unicode_minus'] = False
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.colors import ListedColormap
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

import pickle
from glob import glob
from natsort import natsorted
from tqdm import tqdm

In [ ]:
import sklearn.preprocessing as pp

def cosine_similarity(mat):
    normed_mat = pp.normalize(mat, axis=1)
    return normed_mat.dot(normed_mat.T)

In [ ]:
adata = sc.read_h5ad('../results/adata_proc_singlets.h5ad')
adata

In [ ]:
celltype_key = 'major_celltype'
celltypes = adata.obs[celltype_key].astype(str).unique()
celltypes

In [ ]:
label_mat = adata.obsm['strict_labels']
label_mat

In [ ]:
# number of subgraphs with >= minimum cells
min_cells = 10
idx = np.where(((label_mat != 0).sum(axis=0).A[0] >= min_cells))[0]
idx.shape

In [ ]:
# idx is the index of the subgraph labels; each row here is a group of cells with the same sgRNA+sgUMI+MULTI barcode

cell_idx_stack = []
cell_read_stack = []
ct_stack = []
for k in tqdm(idx):
    cell_idx_stack.append(label_mat[:, k].tocoo().row)
    cell_read_stack.append(label_mat[:, k].tocoo().data)
    
    # map neighbor idx to cell types
    ct_stack.append(adata.obs[celltype_key].iloc[cell_idx_stack[-1]].value_counts().reindex(celltypes).replace(np.nan, 0).astype(int))
    
df = pd.DataFrame({'cell_idx': cell_idx_stack, 'reads': cell_read_stack})
df[celltypes] = pd.DataFrame(ct_stack, index=np.arange(len(df)))  # append cell type counts

In [ ]:
df[celltypes + '_prop'] = df[celltypes] / df[celltypes].sum(axis=1).values[:, None]

df['subgraph_size'] = df['cell_idx'].apply(lambda x: len(x))
df['label'] = labels.iloc[idx].values
df['perturbation'] = df['label'].str.split('_').str[0]
df['tumor'] = df['label'].str.split(':').str[-1]
df['sgRNA'] = df['label'].str.split(':').str[0].str[:-16]

In [ ]:
df.to_csv(f'../results/subgraphs_{celltype_key}_min{min_cells}cells.txt', sep='\t')

In [ ]:
n_subgraphs_per_tumor = df.groupby(['sgRNA', 'tumor']).size().reset_index()
n_subgraphs_per_tumor['tumor_total'] = n_subgraphs_per_tumor['tumor'].map(n_subgraphs_per_tumor.groupby('tumor')[0].sum())
n_subgraphs_per_tumor['proportion'] = n_subgraphs_per_tumor[0] / n_subgraphs_per_tumor['tumor_total']
n_subgraphs_per_tumor

In [ ]:
n_senders_per_tumor = meta[
    (meta['sgRNA_perturbation'] != 'unassigned') & 
    ~meta['tumor'].isna()].groupby([
        'sgRNA_perturbation', 
        'tumor'
]).size().reset_index()

n_senders_per_tumor['tumor_total'] = n_senders_per_tumor['tumor'].map(n_senders_per_tumor.groupby('tumor')[0].sum()).astype(int)
n_senders_per_tumor['proportion'] = n_senders_per_tumor[0] / n_senders_per_tumor['tumor_total']

n_senders_per_tumor

In [ ]:
by_guide = pd.DataFrame(
    {
        'n_subgraphs': df['sgRNA'].value_counts(), 
        'n_senders': meta['sgRNA_perturbation'].value_counts(),
        'prop_subgraphs': n_subgraphs_per_tumor.groupby('sgRNA')['proportion'].mean(),
        'prop_senders': n_senders_per_tumor.groupby('sgRNA_perturbation')['proportion'].mean()
    }
).replace(np.nan, 0)
by_guide['target'] = by_guide.index.str.split('_').str[0]
by_guide['median_size'] = by_guide.index.map(df.groupby('sgRNA')['subgraph_size'].median())
by_guide.drop('unassigned', inplace=True)

by_guide

In [ ]:


with plt.rc_context({'figure.figsize':(5, 5)}):
    sns.scatterplot(data=by_guide, x='prop_senders', y='prop_subgraphs', 
                    color='lightgrey',
                    size='median_size', 
                    sizes=(20, 200),
                    edgecolor='k', lw=.5
                    )
    plt.title('Proportion of communities per labeling cells')
    
    
    # best fit line
    xy = by_guide[['prop_senders', 'prop_subgraphs']]
    x = xy.iloc[:, 0].values
    y = xy.iloc[:, 1].values

    mask = np.any(xy <= 0.0, axis=1)
    lxy = np.log10(xy.loc[~mask, :])

    lx = lxy.values[:, 0]
    ly = lxy.values[:, 1]

    r2, p = pearsonr(lx, ly)
    sig, exp = f"{p:.2e}".split("e")

    # line of best fit
    slope, intercept = np.polyfit(lx, ly, 1)
    
    x_fit = np.linspace(x.min(), x.max(), 100)
    y_fit = 10**intercept * (x_fit ** slope)
    
    # best fit line
    plt.plot(x_fit, y_fit, color='red', label='Best Fit')
    
    # 95% confidence interval
    x_grid_log = np.linspace(lx.min(), lx.max(), 100)
    p = np.polyfit(lx, ly, deg=1)
    y_fit_log = np.polyval(p, x_grid_log)
    n = len(x)
    dof = n - 2
    residuals_log = ly - np.polyval(p, lx)
    s_e = np.sqrt(np.sum(residuals_log**2) / dof)
    
    se_slope = s_e / np.sqrt(np.sum((lx - np.mean(lx))**2))
    slope_ci_lower = slope - t_crit * se_slope
    slope_ci_upper = slope + t_crit * se_slope

    # standard error of the mean prediction (in log space)
    se_fit_log = s_e * np.sqrt(1/n + (x_grid_log - np.mean(lx))**2 / np.sum((lx - np.mean(lx))**2))

    # compute 95% CI bounds in log space
    t_crit = stats.t.ppf(0.975, df=dof)
    ci_lower_log = y_fit_log - t_crit * se_fit_log
    ci_upper_log = y_fit_log + t_crit * se_fit_log

    # transform back to original space
    x_grid = 10**x_grid_log
    y_fit = 10**y_fit_log
    ci_lower = 10**ci_lower_log
    ci_upper = 10**ci_upper_log
    plt.fill_between(x_grid, ci_lower, ci_upper, color="red", alpha=0.2, label="95% CI")

    latex_label = rf"""R={r2:.3f}
                       $P={sig} \times 10^{{{int(exp)}}}$
                       m={slope:.2f} [{slope_ci_lower:.2f}-{slope_ci_upper:.2f}]"""

    ax = plt.gca()
    ax.text(0.95, 0.05, latex_label,
            transform=ax.transAxes,
            horizontalalignment='right', 
            verticalalignment='bottom')
    
    plt.ylabel('proportion of communities')
    plt.xlabel('proportion of sender cells')
    plt.yscale('log')
    plt.xscale('log')
    
plt.tight_layout()
plt.show()    

# show cell type composition

In [ ]:
# each row is a celltype in a subgraph
# group_key = 'perturbation'  # either sgRNA or perturbation (or tumor?)
group_key = 'label'
long_props = df[[group_key] + list(celltypes+'_prop')].melt(id_vars=[group_key])
long_props['variable'] = long_props['variable'].str.split('_prop').str[0]
long_props

In [ ]:
wide_props = long_props.groupby([group_key, 'variable']).mean().reset_index().pivot(index=group_key, columns='variable', values='value')
wide_props

## geosketch subsample
for a manageable subset for plotting

In [ ]:
import geosketch
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(svd_solver='arpack')

In [ ]:
X_pca = pca.fit_transform(wide_props)
X_pca.shape

In [ ]:
N_samples = 5000 

sketch_indices = geosketch.gs(X_pca, N_samples, replace=False, seed=23, verbose=True)

wide_props_subset = wide_props.iloc[sketch_indices]
wide_props_subset.shape

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, cophenet

def full_data_kmeans_stability(data, k, n_iterations=10):
    """
    Measures K-means stability across n initializations without subsampling.
    """
    n_samples = data.shape[0]
    # Store labels for each iteration: shape (n_iterations, n_samples)
    all_labels = np.zeros((n_iterations, n_samples), dtype=int)
    
    
    for i in range(n_iterations):
        # Use a different random state for each run to test initialization stability
        km = KMeans(n_clusters=k, n_init=1, init='k-means++', random_state=i)
        all_labels[i, :] = km.fit_predict(data)

    
    # We define similarity(i, j) = (count of times labels[i] == labels[j]) / n_iterations
    # This is equivalent to the mean of binary connectivity matrices

    # To avoid N^2 memory, we compute the condensed distance matrix (1 - similarity)
    dist_vec = pdist(all_labels.T, metric=lambda u, v: 1 - np.mean(u == v))
    
    Z = linkage(dist_vec, method='average')
    ccc, _ = cophenet(Z, dist_vec)
    
    return ccc, Z

def analyze_stability_sweep(data, k_max, n_iterations=10):
    print(f"Running {n_iterations} iterations of K-means per rank...")
    
    results = []
    
    k_range = np.arange(2, k_max+1)
    for k in tqdm(k_range):
        ccc, _ = full_data_kmeans_stability(data, k, n_iterations)
        results.append({'k': k, 'CCC': ccc})
        
    df_stab = pd.DataFrame(results, index=k_range)
    
    # Plotting code remains the same as previously provided
    return df_stab

In [ ]:
ccc_res = analyze_stability_sweep(wide_props_subset, k_max=20, n_iterations=30)

In [ ]:
# wide_props_subset results, n_iter=30, k_max=20

plt.figure(figsize=(8, 4))
plt.plot(ccc_res['k'], ccc_res['CCC'], 'bo-')
plt.grid()
plt.xlabel('Number of clusters (k)')
plt.ylabel('cophenetic score')
plt.title('Cophenetic Analysis for K-means')
ax = plt.gca()
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

In [ ]:
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, leaves_list, optimal_leaf_ordering
from scipy.spatial.distance import pdist

from sklearn.metrics import silhouette_score
from sklearn_extra.cluster import KMedoids
from scipy.spatial.distance import jensenshannon, pdist, squareform

def nested_reorder_synced(df, n_clusters, method='kmeans', train_idx=None, meta_cluster=None, 
                          return_model=False, order_within_clusters=True, random_state=23):
    assert method in ('kmeans', 'kmedoids')
    
    if method == 'kmeans':
        print('running kmeans')
        km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=100)
        if train_idx:
            km.fit(df.iloc[train_idx])
            labels = km.predict(df)
        else:
            labels = km.fit_predict(df)
    elif method == 'kmedoids':
        print('running kmedoids on JSD')
        km = KMedoids(n_clusters=n_clusters, metric='precomputed', init='k-medoids++', max_iter=300, random_state=random_state)
        print('calculating pairwise distance matrix')
        
        if train_idx:
            dist_matrix = squareform(pdist(df.iloc[train_idx].values, metric=jensenshannon))
            km.fit(dist_matrix)
            
            medoids = kmed.cluster_centers_
            distances_to_medoids = cdist(df.values, medoids, metric=jensenshannon)
            labels = np.argmin(distances_to_medoids, axis=1)
        else:
            dist_matrix = squareform(pdist(df.values, metric=jensenshannon))
            labels = km.fit_predict(dist_matrix)
    
    df_temp = df.copy()
    df_temp['cluster'] = labels

    # determine cluster-level order
    cluster_means = df_temp.groupby('cluster').mean()
    
    if meta_cluster in cluster_means.columns.tolist():
        print(f'ordering by {meta_cluster}')
        ordered_cluster_ids = cluster_means[meta_cluster].sort_values().index.tolist()
    else:
        print('ordering clusters')
        # cluster the clusters!
        meta_dist = pdist(cluster_means.values, metric=jensenshannon)
        meta_linkage = linkage(meta_dist, method='ward')
        ordered_cluster_ids = leaves_list(optimal_leaf_ordering(meta_linkage, meta_dist))
    
    if not order_within_clusters:
        if return_model:
            return ordered_cluster_ids, km
        else:
            return ordered_cluster_ids
    else:
        print('ordering within clusters')
        final_order = []
        for cluster_id in tqdm(ordered_cluster_ids):
            subset = df_temp[df_temp['cluster'] == cluster_id].drop(columns='cluster')

            if len(subset) > 1:
                # Hierarchical clustering within the block
                sub_dist = pdist(subset.values, metric=jensenshannon)
                sub_linkage = linkage(sub_dist, method='ward')
                sub_order = leaves_list(optimal_leaf_ordering(sub_linkage, sub_dist))
                final_order.extend(subset.index[sub_order])
            else:
                final_order.extend(subset.index)
        if return_model:
            return df.loc[final_order], df_temp['cluster'].loc[final_order], ordered_cluster_ids, km
        else:
            return df.loc[final_order], df_temp['cluster'].loc[final_order], ordered_cluster_ids

In [ ]:
n_clust = 7

wide_mean_ordered, labels_ordered, meta_labels_ordered, km = nested_reorder_synced(
    wide_props_subset, 
    n_clust, 
    method='kmeans', 
    return_model=True,
    order_within_clusters=True,
#     meta_cluster='cancer_prop', # order clusters by cancer prop
#     train_idx=sketch_indices
)

In [ ]:
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import matplotlib.transforms as transforms

from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.cluster.hierarchy import dendrogram

In [ ]:
N_samples = len(wide_mean_ordered)

fig, (ax, ax_cluster, ax_archetypes) = plt.subplots(
    3, 1, figsize=(18, 10), 
    gridspec_kw={'height_ratios': [5, 0.5, 5], 'hspace':0.05},
    sharex=True
)


# set colormap
lut = {k: v for k, v in zip(major_celltype_palette.index, major_celltype_palette.values)}
colors = [lut[k] for k in celltype_order]


## draw stacked barplot
print('drawing stacked barplot')
wide_mean_ordered[celltype_order].plot(kind='bar', stacked=True, width=1.0, ax=ax, color=colors, legend=False, rasterized=savefig)

h, l = ax.get_legend_handles_labels()
l = [_.split('_prop')[0] for _ in l]
ax.legend(h, l, bbox_to_anchor=(1.0, 0.5), loc='center left', title='cell type', ncol=2)

ax.set_title('label celltype proportions')
ax.set_ylabel('proportion of label')
ax.set_xlabel(group_key)
ax.set_xticklabels([])
ax.set_xlim(0, len(labels_ordered))
ax.spines[['right', 'top', 'bottom']].set_visible(False)


## draw cluster labels
print('drawing cluster indicator bars')
labels = labels_ordered.loc[wide_mean_ordered.index].values

diff = np.where(labels[1:] != labels[:-1])[0] + 1
cluster_boundaries = np.concatenate(([0], diff, [len(labels)]))

ax_cluster.clear()

# draw a rectangle for each cluster block
for i in range(len(cluster_boundaries) - 1):
    start = cluster_boundaries[i]
    end = cluster_boundaries[i+1]
    midpoint = (start + end) / 2
    cluster_id = labels[start]

    rect = patches.Rectangle((start, 0), end - start, 1, 
                             facecolor=color29[1:][i],
                             edgecolor='none',
                             transform=ax_cluster.get_xaxis_transform())
    
    ax_cluster.add_patch(rect)
    
    # Draw the text
    ax_cluster.text(x=midpoint, y=0.5, s=f"C{cluster_id}", color='black',
        fontsize=9, fontweight='bold', 
        ha='center', va='center', clip_on=True)

ax_cluster.set_xlim(0, len(labels_ordered))
ax_cluster.set_ylim(0, 1)
ax_cluster.set_ylabel('cluster')
ax_cluster.set_axis_off()
# ----------------------------------------------------------------



## draw heatmap
print('drawing mean proportions')
# --- 1. Calculate Z-scores for the Archetypes ---
cluster_archetypes = wide_mean_ordered.assign(
    cluster=labels # use the reordered labels
).groupby('cluster').mean().loc[meta_labels_ordered]

celltype_labels = cluster_archetypes.columns
n_celltypes = len(celltype_labels)
n_archetypes = len(meta_labels_ordered)

cluster_archetypes_z = (cluster_archetypes - cluster_archetypes.mean(axis=0)) / cluster_archetypes.std(axis=0)
cluster_archetypes_z = cluster_archetypes_z.fillna(0)

y_dist = pdist(cluster_archetypes_z.T, metric='euclidean')
y_linkage = linkage(y_dist, method='ward')
y_order = leaves_list(optimal_leaf_ordering(y_linkage, y_dist))

cluster_archetypes_z = cluster_archetypes_z.iloc[:, y_order]

# center cmap
cmap_z = plt.get_cmap('RdBu_r')
v_limit = max(abs(cluster_archetypes_z.values.min()), abs(cluster_archetypes_z.values.max()))

# draw circles
trans = transforms.blended_transform_factory(ax_archetypes.transAxes, ax_archetypes.transData)
x_pos_fixed = -0.1

for i, current_cluster_id in enumerate(cluster_archetypes_z.index):
    z_data = cluster_archetypes_z.loc[current_cluster_id]
    
    start = cluster_boundaries[i]
    end = cluster_boundaries[i+1]
    
    for j, ct_name in enumerate(cluster_archetypes_z.columns):
        z_val = z_data.iloc[j]
        ypos = n_celltypes - j - 1
        
        # map value to cmap
        color_idx = (z_val + v_limit) / (2 * v_limit)
        color = cmap_z(np.clip(color_idx, 0, 1))
        
        rect = patches.Rectangle(
            (start, ypos),
            end - start,
            1,
            facecolor=color,
            edgecolor='white',
            linewidth=0.5
        )
        ax_archetypes.add_patch(rect)
        
        if i == 0:
            circle_color = lut[ct_name]
            ax_archetypes.plot(x_pos_fixed, ypos + 0.5, 
                       marker='o', 
                       markersize=10, 
                       color=circle_color, 
                       markeredgecolor='black',
                       markeredgewidth=0.5,
                       transform=trans, 
                       clip_on=False,
                       linestyle='none')

yticks_pos = [n_celltypes - j - 0.5 for j in range(n_celltypes)]
ytick_names = cluster_archetypes_z.columns.tolist()

ax_archetypes.set_yticks(yticks_pos)
ax_archetypes.set_yticklabels(ytick_names, fontsize=9)

# adjust spacing for yticks
ax_archetypes.tick_params(axis='y', length=5, left=True, labelleft=True)
ax_archetypes.spines['left'].set_visible(True)

ax_archetypes.spines[['right', 'top', 'bottom']].set_visible(False)
ax_archetypes.xaxis.set_visible(False)
            
            
# plot dendrogram
pos_heatmap = ax_archetypes.get_position()

dendro_width = 0.06
offset_points = 20

offset_fig = offset_points / (fig.dpi * fig.get_size_inches()[0])
x_start = pos_heatmap.x1 + offset_fig

ax_dendro_right = fig.add_axes([x_start, pos_heatmap.y0, dendro_width, pos_heatmap.height])

with plt.rc_context({'lines.linewidth': 0.8}):
    dendrogram(
        y_linkage, 
        ax=ax_dendro_right, 
        orientation='right', 
        no_labels=True,
        color_threshold=0, 
        above_threshold_color='black'
    )
ax_dendro_right.set_axis_off()
ax_dendro_right.invert_yaxis()

# update cbar
norm_z = mcolors.Normalize(vmin=-v_limit, vmax=v_limit)
sm_z = plt.cm.ScalarMappable(cmap=cmap_z, norm=norm_z)
sm_z.set_array([])
cbar_ax = fig.add_axes([1.01, 0.1, 0.015, 0.25])
fig.colorbar(sm_z, cax=cbar_ax, label='Z-score (Mean Proportion)')


plt.tight_layout()
plt.show()

# partitioning and co-localization

In [ ]:
mat = adata.obsm['strict_labels'].copy()

# membership matrix
mem = mat.copy()
mem.data[:] = 1

# number of shared barcodes
adj = mem.dot(mem.T)

In [ ]:
def sp_zero_diag(mat):
    return mat - sparse.dia_matrix((mat.diagonal()[np.newaxis, :], [0]), shape=mat.shape)

In [ ]:


# calculate connected subgraphs
adj = sp_zero_diag(adj)

_, comm_assign = sparse.csgraph.connected_components(adj)
valid_subgraphs = pd.Series(comm_assign).value_counts()
valid_subgraphs = valid_subgraphs[valid_subgraphs > 1]

In [ ]:
# just calculate the fuzzy simplicial set
# https://github.com/scverse/scanpy/blob/main/src/scanpy/neighbors/_connectivity.py#L103
from sklearn.neighbors import NearestNeighbors
from umap.umap_ import fuzzy_simplicial_set

k = 10
nn_metric = 'cosine'
partition_type = leidenalg.RBConfigurationVertexPartition

outdir = '../results/fuzzy_graphs'
os.makedirs(outdir, exist_ok=True)

leiden_res = []
for i, subg_idx in enumerate(valid_subgraphs.index):
    print(f'{i+1} / {len(valid_subgraphs)}')
    cell_idx = np.where(comm_assign == subg_idx)[0]
    n = len(cell_idx)
    
    outpath_connectivities = os.path.join(outdir, f'UMAP_fuzzy_{nn_metric}_connectivities_k{k}_{subg_idx}_n{n}.npz')
    outpath_distances = os.path.join(outdir, f'UMAP_fuzzy_{nn_metric}_distances_k{k}_{subg_idx}_n{n}.npz')
    outpath_leiden = os.path.join(outdir, f'UMAP_fuzzy_{nn_metric}_leiden_k{k}_{subg_idx}_n{n}.txt')
    
    # set data matrix
    X = adata.obsm['strict_labels'][cell_idx,].log1p()
    
    print('calculating kNN and fuzzy simplicial graph')
    nn = NearestNeighbors(n_neighbors=k+1, metric=nn_metric).fit(X)
    knn_indices = nn.kneighbors_graph(X, mode='connectivity').indices.reshape(X.shape[0], k+1)
    knn_dists = nn.kneighbors_graph(X, mode='distance').data.reshape(X.shape[0], k+1)

    # exclude self
    knn_indices = knn_indices[:, 1:]
    knn_dists = knn_dists[:, 1:]
    
    W, sigmas, rhos, dists = fuzzy_simplicial_set(
        X=X, 
        n_neighbors=k, 
        random_state=23, 
        metric=nn_metric,
        knn_indices=knn_indices,
        knn_dists=knn_dists,
        return_dists=True
    )

    W = W.tocsr()
    dists = dists.tocsr()
    
    # save results
    sparse.save_npz(outpath_connectivities, W)
    sparse.save_npz(outpath_distances, dists)
    
    # leiden clustering
    sources, targets = W.nonzero()
    g = ig.Graph(directed=None)
    g.add_vertices(W.shape[0])
    g.add_edges(list(zip(sources, targets)))
    
    g.es['weight'] = W.data
    part = leidenalg.find_partition(g, partition_type, resolution_parameter=1)
    leiden_comms = pd.Series(part.membership, index=adata.obs.iloc[cell_idx].index)
    tumor_id = adata.obs.iloc[cell_idx]['tumor'].iloc[0]
    leiden_comms = tumor_id + '-' + leiden_comms.astype(str)
    print(f'found {len(leiden_comms.unique())} leiden partitions')
    leiden_comms.to_csv(outpath_leiden, sep='\t')
    
    leiden_res.append(leiden_comms)
    print()
leiden_res = pd.concat(leiden_res)

In [ ]:
adata.obs['spatial_leiden'] = leiden_res.reindex(adata.obs.index)

In [ ]:
celltype_key = 'fine_celltype'

sp_counts = adata.obs.groupby('spatial_leiden')[celltype_key].value_counts().reset_index().pivot(
    index='spatial_leiden', 
    columns=celltype_key, 
    values='count'
)
sp_props = sp_counts / sp_counts.sum(axis=1).values.reshape(-1, 1)

In [ ]:
corrs = {}
for t in adata.obs['tumor'].dropna().unique():
    tmp = sp_props.loc[sp_props.index.str.startswith(t)]
    corrs[t] = tmp.corr().fillna(0).reindex(sp_props.columns).T.reindex(sp_props.columns)
    

In [ ]:
mean_corr = np.mean([c.values for c in corrs.values()], axis=0)
mean_corr = pd.DataFrame(mean_corr, index=sp_props.columns, columns=sp_props.columns)
mean_corr

In [ ]:
plt.hist(mean_corr.values.ravel(), bins=200)
plt.axvline(-0.3, ls='--', lw=1, color='k')
plt.axvline(0.3, ls='--', lw=1, color='k')

plt.show()

In [ ]:
stack = np.stack([c.values for c in corrs.values()])
pval = np.apply_along_axis(lambda x: ttest_1samp(x, popmean=0).pvalue, axis=0, arr=stack)
pval = pd.DataFrame(pval, index=sp_props.columns, columns=sp_props.columns)
signed_logpval = np.sign(mean_corr) * -np.log10(pval)

In [ ]:
tmp = mean_corr.copy()
np.fill_diagonal(tmp.values, 0)

cmap = plt.cm.RdBu_r
g = sns.clustermap(
    tmp,
    figsize=(18, 18),
    dendrogram_ratio=(0.1, 0.1),
    cbar_pos=(0.2, 0.2, 0.05, 0.2),
    method='complete',
    vmin=-.3, vmax=0.3,
    xticklabels=False,
    center=0, cmap=cmap, square=True,
)

ax_hist = g.ax_cbar.twiny()
ax_hist.set_position(g.ax_cbar.get_position())

# plot histogram on cbar
idx = np.tril_indices_from(tmp.values, k=-1)
vals = tmp.values[idx].flatten()

sns.histplot(y=vals, 
             ax=ax_hist, 
             color="black", 
             element="step", 
             alpha=0.2,
             kde=True)

ax_hist.set_xlim(ax_hist.get_xlim()[::-1])
ax_hist.set_facecolor('none')
for spine in ["top", "right", "bottom"]:
    ax_hist.spines[spine].set_visible(False)

ax_hist.set_xlabel("Frequency", fontsize=8)
g.ax_cbar.yaxis.set_label_position("right")
g.ax_cbar.yaxis.tick_right()
g.ax_cbar.set_title("Distribution of R")
g.ax_cbar.set_ylabel("Mean Pearson R", fontsize=10)

# mask tril
mask = np.tril(np.ones_like(tmp))
values = g.ax_heatmap.collections[0].get_array().reshape(tmp.shape)
new_values = np.ma.array(values, mask=mask)
g.ax_heatmap.collections[0].set_array(new_values)
g.ax_row_dendrogram.set_visible(False)

plt.show()

In [ ]:
tmp = signed_logpval.copy(); statistic = 'P-value'

np.fill_diagonal(tmp.values, 0)


cmap = plt.cm.PuOr_r
g2 = sns.clustermap(
    tmp,
    figsize=(18, 18),
    dendrogram_ratio=(0.1, 0.1),
    cbar_pos=(0.8, 0.6, 0.05, 0.2),
    row_linkage=g.dendrogram_row.linkage,
    col_linkage=g.dendrogram_col.linkage,
    row_cluster=True,
    col_cluster=True,
    vmin=-4, vmax=4,
    yticklabels=False,
    center=0, cmap=cmap
)

ax_hist = g2.ax_cbar.twiny()
ax_hist.set_position(g2.ax_cbar.get_position())
idx = np.tril_indices_from(tmp.values, k=-1)
vals = tmp.values[idx].flatten()

sns.histplot(y=vals, 
             ax=ax_hist, 
             color="black", 
             element="step", 
             alpha=0.2,
             kde=True)

ax_hist.set_xlim(ax_hist.get_xlim()[::-1])
ax_hist.set_facecolor('none')
for spine in ["top", "right", "bottom"]:
    ax_hist.spines[spine].set_visible(False)


ax_hist.set_xlabel("Frequency", fontsize=8)
g2.ax_cbar.yaxis.set_label_position("right")
g2.ax_cbar.yaxis.tick_right()
g2.ax_cbar.set_title(f"Distribution of {statistic} (-log10)")
g2.ax_cbar.set_ylabel(f"{statistic} (-log10)", fontsize=10)


mask = np.triu(np.ones_like(tmp))
values = g2.ax_heatmap.collections[0].get_array().reshape(tmp.shape)
new_values = np.ma.array(values, mask=mask)
g2.ax_heatmap.collections[0].set_array(new_values)
g2.ax_row_dendrogram.set_visible(False)

plt.show()